# LASSO-type sparse priors for the M5 covariates, with fast inference

Issue #155, part of the #143 re-plan (references in #144).

**Question.** Per-series coefficients under Gaussian priors did not help on the full M5 run: stage 1 (#146) scored
0.86564 against 0.79563 on pre-validation. The covariates are mostly event and calendar dummies with little signal on
intermittent series, so a prior that sets most of them to zero, or close to it, is the next thing to try. Which
approach is worth building as an M5 entry, and what does it need that is not there yet?

**Method.** Everything is scored on the 3-item × 10-store benchmark subset of #154 (30 series, `m5_subset.py`) at the
pre-validation origin, by the 12-level WRMSSE of `m5_score_submission.py`, with the fit time beside it. No full run is
made. All approaches share the #118 default's states (a level and six weekday offsets), its Yeo-Johnson target and its
half-normal sigma priors, and add the same 73 covariates of `CALENDAR_COVARIATES` (2 annual Fourier terms, 30 events, 11
months, 30 days of the month) as a static regression `X beta` outside the filter, as in `SspReg`:

| group | approach | inference | what it tests |
|---|---|---|---|
| baseline | #118 default | MAP (Adam), no covariates | the score to beat |
| baseline | `SspReg`, Gaussian `tau` = 1 and 0.1 | NUTS | the covariates without sparsity |
| optimisation only | L1 and group L1 | MAP (Adam) + a proximal soft-threshold | exact zeros, in seconds |
| sampling | Laplace (Bayesian LASSO) prior | NUTS | LASSO as a posterior |
| sampling | regularized horseshoe prior | NUTS | heavier tails than the Laplace |
| approximate | the same two priors | SVI, `AutoNormal` | the fast approximation |

**Runtime.** On a laptop CPU the two NUTS fits of the sparse priors take 38 and 26 minutes, and the two `SspReg` fits 10 and 3;
the rest of the notebook runs in about 5 minutes. The cost of NUTS is one of the findings. To rerun it quickly, lower
`num_warmup`, `num_samples` and `num_chains` in `fit_sparse_nuts`; the scores then differ.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, SVI, Predictive, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal

warnings.filterwarnings("ignore")

# The M5 scripts live in playground/m5_prototype and import each other by name.
m5_dir = next(p / "m5_prototype" for p in [Path.cwd(), *Path.cwd().parents] if (p / "m5_prototype").is_dir())
sys.path.insert(0, str(m5_dir))

from m5_common import load_m5_ds  # noqa: E402
from m5_models import CALENDAR_COVARIATES, DEFAULT_MODEL, HALF_NORMAL_PRIORS, MODELS, SspReg  # noqa: E402
from m5_score_submission import compute_hierarchical_wrmsse, window_masks  # noqa: E402
from m5_subset import SUBSET_ITEMS, per_item_wrmsse, subset_series_idx  # noqa: E402

from bunobee.models.m5.m5_ssp_optim import _softplus, _softplus_inv, kalman_filter_1d_batch  # noqa: E402
from bunobee.models.m5.m5_ssp_reg import predict_batch_series_reg  # noqa: E402
from bunobee.models.m5.transforms import PowerTransform  # noqa: E402
from bunobee.regression import make_periodic_dummies  # noqa: E402

print("numpyro", numpyro.__version__, "| jax", jax.__version__, "| devices", jax.devices())

## Data, and the score

`score` is the pre-validation WRMSSE of the subset: the mean over the 12 levels, as `m5_run.py` reports it, plus the
score of each item alone (`high` is `FOODS_3_586`, `mid` `HOUSEHOLD_1_069`, `low` `HOBBIES_1_197`), since the high item
holds most of the dollar sales. The covariate groups are kept so that the group penalty can use them.

In [ ]:
ds = load_m5_ds()
series_idx = subset_series_idx(ds)  # 3 items x 10 stores, item by item (high, mid, low)
fit_mask, forecast_mask = window_masks(ds["split"].values, "prevalidation")
base_ids = ds["series_id"].values[series_idx]
sales = ds["sales"].values[series_idx][:, fit_mask]  # (30, n_fit_days)
n_series, n_fit = sales.shape
print(f"{n_series} series, {n_fit} fit days, {forecast_mask.sum()} forecast days;  items: {SUBSET_ITEMS}")

# The design of every model below: the #118 states (level + six weekday offsets) and the stage-1 calendar covariates.
weekday = ds["day_of_week"].values
Z_all = make_periodic_dummies(weekday.size, period=7, phase=int(weekday[0]), intercept=True)  # (n_dates, 7)
X_all = CALENDAR_COVARIATES.design(ds)  # (n_dates, K)
n_events = X_all.shape[1] - 2 - 11 - 30
GROUPS = {  # column slices of X by covariate group
    "annual": slice(0, 2),
    "events": slice(2, 2 + n_events),
    "month": slice(2 + n_events, 13 + n_events),
    "day_of_month": slice(13 + n_events, X_all.shape[1]),
}
group_of = np.empty(X_all.shape[1], dtype=object)
for name, cols in GROUPS.items():
    group_of[cols] = name
Z, X = Z_all[fit_mask], X_all[fit_mask]
Z_future, X_future = Z_all[forecast_mask], X_all[forecast_mask]
K = X.shape[1]
print(f"Z {Z.shape}, X {X.shape};  columns per group: { {n: len(range(*c.indices(K))) for n, c in GROUPS.items()} }")


def score(forecasts, window="prevalidation"):
    """WRMSSE: the mean over the 12 levels of the subset's forecasts, and the score of each item alone."""
    f = np.asarray(forecasts, dtype=np.float64)
    overall = compute_hierarchical_wrmsse(ds, f, base_ids, window=window)["wrmsse"].mean()
    items = per_item_wrmsse(ds, f, base_ids, window=window)
    return float(overall), items


RESULTS = []


def record(name, forecasts, seconds, **extra):
    """Score a forecast, keep a row for the final table and return it."""
    overall, items = score(forecasts)
    per_item = {f"wrmsse_{tier}": v for tier, v in zip(items["tier"], items["wrmsse"])}
    row = {"approach": name, "wrmsse": overall, **per_item, "seconds": seconds, **extra}
    RESULTS.append(row)
    return row

## What numpyro 0.22 and `bunobee` have for this

Before fitting anything: which of the pieces the issue lists exist. Two do not, and they decide which of the Gibbs
routes below are possible without new code.

In [ ]:
import numpyro.infer as ninf
import numpyro.infer.hmc_gibbs as hmc_gibbs
import numpyro.infer.autoguide as autoguide
import numpyro.infer.reparam as reparam
import bunobee.models.ssp.kalman_1d as kalman_1d

inventory = {
    "dist.Laplace (Bayesian LASSO prior)": hasattr(dist, "Laplace"),
    "dist.HalfCauchy / dist.InverseGamma (horseshoe)": hasattr(dist, "HalfCauchy") and hasattr(dist, "InverseGamma"),
    "dist.InverseGaussian (Park & Casella Gibbs update)": hasattr(dist, "InverseGaussian"),
    "infer.HMCGibbs": hasattr(ninf, "HMCGibbs"),
    "infer.Gibbs / hmc_gibbs.CustomGibbs": hasattr(ninf, "Gibbs") and hasattr(hmc_gibbs, "CustomGibbs"),
    "infer.DiscreteHMCGibbs / MixedHMC / DiscreteGibbs": all(
        hasattr(ninf, n) for n in ("DiscreteHMCGibbs", "MixedHMC", "DiscreteGibbs")
    ),
    "infer.reparam.LocScaleReparam": hasattr(reparam, "LocScaleReparam"),
    "autoguide.AutoNormal / AutoLaplaceApproximation": hasattr(autoguide, "AutoNormal")
    and hasattr(autoguide, "AutoLaplaceApproximation"),
    "bunobee kalman_rts_smoother_1d / kalman_dk_smoother_1d (means only)": hasattr(kalman_1d, "kalman_rts_smoother_1d")
    and hasattr(kalman_1d, "kalman_dk_smoother_1d"),
    "bunobee simulation smoother (draws)": hasattr(kalman_1d, "kalman_simulation_smoother_1d"),
    "bunobee full-covariance filter": hasattr(kalman_1d, "kalman_filter_full_cov"),
}
print(pd.Series(inventory, name="available").to_string())

## The scores to beat

The #118 default is the MAP fit with no covariates. `SspReg` adds the covariates under Gaussian priors and samples them with
NUTS, 10 series per run, with its defaults of 200 warmup draws, 200 draws and 4 chains.

In [ ]:
transform = PowerTransform("yeo-johnson")  # the transform of the #118 default and of SspReg, fitted once per series

# 1. The #118 default: level + weekday states, MAP, no covariates.
t0 = time.time()
f_default, _ = MODELS[DEFAULT_MODEL].fit_forecast(ds, series_idx, fit_mask, forecast_mask)
t_default = time.time() - t0
print(record("#118 default (MAP, no covariates)", f_default, t_default, nonzero=0))

In [ ]:
# 2. SspReg: the same states plus a static regression X beta with Gaussian priors, sampled with NUTS (10 series
# per run).
rows_gauss = {}
for tau in (1.0, 0.1):
    t0 = time.time()
    f, fitted = SspReg(covariates=CALENDAR_COVARIATES, tau=tau).fit_forecast(ds, series_idx, fit_mask, forecast_mask)
    rows_gauss[tau] = f
    print(
        record(
            f"SspReg, Gaussian tau={tau} (NUTS)",
            f,
            time.time() - t0,
            nonzero=K,
            n_eff_min=float(fitted["n_eff_min"].min()),
        )
    )

## Optimisation only: L1 and group L1 as a proximal step

MAP under a Laplace prior has no exact zeros if the optimiser only follows its gradient, so the L1 term is handled by
a proximal step after each optax update: `beta <- soft_threshold(beta, thr)`. Adam is a gradient step in the diagonal
metric `D = 1 / (sqrt(nu_hat) + eps)`, so the matching threshold is `lr * lambda * D`, not `lr * lambda`. A threshold of a plain
`lr * lambda` would be in the wrong units, as Adam's step is `lr` per coefficient whatever the scale of the gradient. With the metric, a coefficient is non-zero at a fixed
point only if its data gradient is above `lambda`, in units of log-likelihood per unit of coefficient.

The group version thresholds each covariate group (annual, events, month, day of month) as a block, so a group is
either in or out. `lambda` is chosen by the mean RMSSE of the forecast of three rolling 28-day folds at the end of the fit
window, so it never looks at the origin being scored.

In [ ]:
def soft_threshold(beta, thr):
    """Prox of ``thr * |beta|_1``: shrinks each coefficient toward 0 by ``thr`` and sets the small ones to exactly 0."""
    return jnp.sign(beta) * jnp.maximum(jnp.abs(beta) - thr, 0.0)


def group_soft_threshold(beta, thr):
    """Prox of ``thr * sum_g sqrt(|g|) ||beta_g||_2`` over the covariate groups: a whole group goes to 0 together.

    ``thr`` is per coefficient (see ``fit_map_l1``); a group uses the mean of its coefficients' thresholds.
    """
    out = beta
    for cols in GROUPS.values():
        block = beta[:, cols]
        norm = jnp.linalg.norm(block, axis=1, keepdims=True)
        group_thr = thr[:, cols].mean(axis=1, keepdims=True) * jnp.sqrt(block.shape[1])
        out = out.at[:, cols].set(block * jnp.maximum(1.0 - group_thr / jnp.maximum(norm, 1e-12), 0.0))
    return out


def fit_map_l1(sales, Z, X, lam, prox=soft_threshold, tau=None, n_iter=300, lr=3e-2, sigma_priors=HALF_NORMAL_PRIORS):
    """MAP fit of B series with an L1 penalty on the static coefficients, by Adam followed by a proximal step.

    Minimises ``-log p(y | sigmas, beta) - log p(sigmas) + lam * |beta|_1`` per series, where the likelihood is the
    Kalman filter's on ``y - X beta``. Adam is a gradient step in the metric ``D = 1 / (sqrt(nu_hat) + eps)``
    (``nu_hat`` is its bias-corrected second moment), so the matching proximal step soft-thresholds each coefficient
    by ``lr * lam * D``: a coefficient is non-zero at a fixed point only if its data gradient exceeds ``lam``,
    whatever ``D`` is. A threshold of a plain ``lr * lam`` would be in the wrong units, as Adam's step is ``lr`` per
    coefficient whatever the scale of the gradient. A coefficient can reach exactly 0.

    ``tau`` adds a Gaussian prior ``N(0, tau^2)`` on ``beta`` to the smooth loss, for the ridge comparison (use
    ``lam=0`` with it). Returns a dict shaped like ``fit_batch_series_reg``'s, with one "draw", so that
    ``predict_batch_series_reg`` forecasts from it.
    """
    sigma_h_prior, sigma_q_prior = sigma_priors["sigma_h_prior"], sigma_priors["sigma_q_prior"]
    params_t = transform.fit(sales)
    y = jnp.asarray(transform.forward(sales, params_t))
    B, n_states = y.shape[0], Z.shape[1]
    Zj, Xj = jnp.asarray(Z, dtype=jnp.float32), jnp.asarray(X, dtype=jnp.float32)

    def neg_log_post(raw, beta):
        sigma_h, sq_level, sq_dow = (_softplus(raw[:, i]) for i in range(3))
        sigma_q = jnp.column_stack([sq_level, jnp.broadcast_to(sq_dow[:, None], (B, n_states - 1))])
        lp, *_ = kalman_filter_1d_batch(
            a0=jnp.zeros(n_states),
            P0=jnp.ones(n_states),
            Z=Zj,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y - beta @ Xj.T,
            logp=True,
            chunk_size=None,
        )
        log_prior = (
            sigma_h_prior.distribution.log_prob(sigma_h)
            + sigma_q_prior.distribution.log_prob(sq_level)
            + sigma_q_prior.distribution.log_prob(sq_dow)
        )
        if tau is not None:
            log_prior = log_prior + dist.Normal(0.0, tau).log_prob(beta).sum(axis=1)
        return -(lp + log_prior).sum()

    raw = jnp.tile(jnp.array([_softplus_inv(sigma_h_prior.init), *[_softplus_inv(sigma_q_prior.init)] * 2]), (B, 1))
    beta = jnp.zeros((B, Xj.shape[1]))
    opt = optax.adam(lr)
    state = opt.init((raw, beta))

    @jax.jit
    def step(raw, beta, state):
        loss, grads = jax.value_and_grad(neg_log_post, argnums=(0, 1))(raw, beta)
        updates, state = opt.update(grads, state)
        raw, beta = optax.apply_updates((raw, beta), updates)
        adam = state[0]  # ScaleByAdamState
        nu_hat = adam.nu[1] / (1.0 - 0.999**adam.count)
        return raw, prox(beta, lr * lam / (jnp.sqrt(nu_hat) + 1e-8)), state, loss

    losses = []
    for _ in range(n_iter):
        raw, beta, state, loss = step(raw, beta, state)
        losses.append(float(loss))
    sigma_h, sq_level, sq_dow = (np.asarray(_softplus(raw[:, i]))[None] for i in range(3))
    return {
        "samples": {
            "sigma_h": sigma_h,
            "sigma_q_level": sq_level,
            "sigma_q_dow": sq_dow,
            "beta": np.asarray(beta)[None],
        },
        "y": np.asarray(y),
        "transform_params": params_t,
        "transform": transform,
        "Z": Zj,
        "X": np.asarray(Xj),
        "losses": losses,
    }

In [ ]:
HOLDOUT = 28  # days per fold
CUTS = [n_fit - k * HOLDOUT for k in (3, 2, 1)]  # three rolling origins at the end of the fit window: no look-ahead


def holdout_rmsse(forecast, actual, train):
    """RMSSE of a holdout forecast per series, scaled by the in-sample one-step naive error of the training days."""
    scale = np.maximum(np.mean(np.diff(train, axis=1) ** 2, axis=1), 1e-8)
    return np.sqrt(np.mean((forecast - actual) ** 2, axis=1) / scale)


def choose_lambda(prox, lams, **kw):
    """Pick the lambda of the lowest holdout RMSSE, averaged over the series and over three rolling 28-day folds."""
    out = {}
    for lam in lams:
        errs = []
        for cut in CUTS:
            res = fit_map_l1(sales[:, :cut], Z[:cut], X[:cut], lam, prox=prox, **kw)
            pred = predict_batch_series_reg(res, Z[cut : cut + HOLDOUT], X[cut : cut + HOLDOUT], n_draws=1)
            errs.append(np.mean(holdout_rmsse(pred, sales[:, cut : cut + HOLDOUT], sales[:, :cut])))
        out[lam] = float(np.mean(errs))
    return min(out, key=out.get), out


LAMBDAS = (
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0,
    300.0,
)  # nats per unit of |beta|: a data gradient below lambda leaves a coefficient at 0
# Control: the same fitter with every coefficient held at 0 (lambda -> infinity), so a difference to the #118 default
# row is the fitter (300 steps against 100, one-point forecast of the transformed fit), not the covariates.
t0 = time.time()
res0 = fit_map_l1(sales, Z, X, 1e9)
f0 = predict_batch_series_reg(res0, Z_future, X_future, n_draws=1)
print(record("MAP, own fitter, no covariates (lambda=inf)", f0, time.time() - t0, nonzero=0))

prox_rows = {}
for label, prox in (("L1", soft_threshold), ("group L1", group_soft_threshold)):
    t0 = time.time()
    lam, holdout = choose_lambda(prox, LAMBDAS)
    t_select = time.time() - t0
    t0 = time.time()
    res = fit_map_l1(sales, Z, X, lam, prox=prox)
    f = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
    t_fit = time.time() - t0
    beta = res["samples"]["beta"][0]
    prox_rows[label] = (f, res)
    print(f"{label}: holdout RMSSE by lambda {holdout} -> lambda={lam:g}")
    print(
        record(
            f"MAP + {label} prox (lambda={lam:g})",
            f,
            t_fit,
            nonzero=float((beta != 0).sum(axis=1).mean()),
            seconds_with_selection=t_select + t_fit,
        )
    )

In [ ]:
# Sensitivity of the pre-validation score to lambda. This reads the score of the evaluation origin, so it only shows
# how flat the choice is: the row above, chosen on the holdout, is the one to compare.
grid = []
for lam in LAMBDAS:
    t0 = time.time()
    res = fit_map_l1(sales, Z, X, lam)
    f = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
    grid.append(
        {
            "lambda": lam,
            "wrmsse": score(f)[0],
            "nonzero_per_series": float((res["samples"]["beta"][0] != 0).sum(axis=1).mean()),
            "seconds": time.time() - t0,
        }
    )
print(pd.DataFrame(grid).to_string(index=False))

The grid above scores each `lambda` at the origin being scored, so it shows the sensitivity of the choice, not a
choice. It is not monotone: a light penalty (`lambda` ≤ 3) helps, a middle one (10-30) keeps the few coefficients with
the biggest in-sample gradient and does worse than keeping none, and a heavy one is the baseline.

### Controls: is it the sparsity, or just any shrinkage?

The L1 rows improve on the default, but so might any MAP fit with the covariates. The next rows use no penalty at all and
the Gaussian priors of `SspReg` on the same MAP fitter. None of them has a coefficient at 0.

In [ ]:
# Controls for "is it the sparsity or just any shrinkage?": the same MAP fitter with no penalty at all, and with the
# Gaussian priors of SspReg (ridge). Neither has a coefficient at exactly 0.
for label, kw in (
    ("no penalty (lambda=0)", dict(lam=0.0)),
    ("Gaussian tau=1 (ridge)", dict(lam=0.0, tau=1.0)),
    ("Gaussian tau=0.1 (ridge)", dict(lam=0.0, tau=0.1)),
):
    t0 = time.time()
    res = fit_map_l1(sales, Z, X, **kw)
    f = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
    print(
        record(f"MAP, {label}", f, time.time() - t0, nonzero=float((res["samples"]["beta"][0] != 0).sum(axis=1).mean()))
    )

### The same MAP fits at the other two origins

One 28-day window on 30 series is a small sample. The MAP fits are cheap, so the same grid is scored at the validation
and evaluation origins too. This is a robustness check of the MAP rows only; the NUTS fits are not repeated, as one costs
up to 40 minutes.

In [ ]:
# The MAP fits take seconds, so the same lambda grid can be scored at the other two origins of the competition too.
# This is a check that the pre-validation table is not one lucky 28-day window; the NUTS fits are not repeated.
origin_rows = []
for window in ("prevalidation", "validation", "evaluation"):
    fm, fcm = window_masks(ds["split"].values, window)
    if not np.isfinite(ds["sales"].values[series_idx][:, fcm]).all():
        print(f"{window}: no actuals in the dataset, skipped")
        continue
    sales_w = ds["sales"].values[series_idx][:, fm]
    base, _ = MODELS[DEFAULT_MODEL].fit_forecast(ds, series_idx, fm, fcm)
    origin_rows.append({"origin": window, "approach": "#118 default", "wrmsse": score(base, window)[0], "nonzero": 0})
    variants = [(f"MAP L1, lambda={lam:g}", dict(lam=lam)) for lam in (0.0, *LAMBDAS, 1e9)]
    variants += [("MAP Gaussian tau=1", dict(lam=0.0, tau=1.0)), ("MAP Gaussian tau=0.1", dict(lam=0.0, tau=0.1))]
    for label, kw in variants:
        res = fit_map_l1(sales_w, Z_all[fm], X_all[fm], **kw)
        f = predict_batch_series_reg(res, Z_all[fcm], X_all[fcm], n_draws=1)
        nz = float((res["samples"]["beta"][0] != 0).sum(axis=1).mean())
        origin_rows.append({"origin": window, "approach": label, "wrmsse": score(f, window)[0], "nonzero": nz})
table = pd.DataFrame(origin_rows).pivot(index="approach", columns="origin", values="wrmsse")
print(table[["prevalidation", "validation", "evaluation"]].round(4).to_string())

## Sampling: a sparse prior on `beta` in `ssp_reg_model`

`beta` is already a NUTS site outside the filter, so the sparse priors are a change of its prior, and the filter keeps its
seven states. Both are written in non-centered form, which the horseshoe needs and which also helps the Laplace:

* **Laplace (Bayesian LASSO):** `beta_k = b * u_k`, `u_k ~ Laplace(0, 1)`, with a per-series scale `b ~ HalfNormal(1)`.
* **Regularized horseshoe** (Piironen & Vehtari): `beta_k = z_k * tau * lam~_k`, `z_k ~ N(0, 1)`, `tau ~ HalfCauchy(0.1)`,
  `lam_k ~ HalfCauchy(1)`, with a slab `c^2 ~ InvGamma(2, 2)` that keeps the large coefficients from being unregularized.
  `numpyro.infer.reparam.LocScaleReparam` would do the non-centering `z * scale` for us; it is written out here to keep
  the model one function.

`fit_sparse_nuts` fits in runs of 10 series with 200 warmup draws, 200 draws and 4 chains, as `SspReg` does, and forecasts with
`predict_batch_series_reg` on 100 posterior draws. A MAP fit under a horseshoe prior is not defined (its density is
unbounded at 0), which is why the horseshoe only appears with a sampler or SVI. SVI (`AutoNormal`, mean-field) is the fast
approximate option: one run over all 30 series.

In [ ]:
def ssp_sparse_model(y, Z, X, prior, sigma_h_prior, sigma_q_prior):
    """``ssp_reg_model`` with a sparsity-inducing prior on ``beta``, all in non-centered form.

    ``prior``:
      * ``"laplace"``: Bayesian LASSO, ``beta_k = b * u_k`` with ``u_k ~ Laplace(0, 1)`` and a per-series scale
        ``b ~ HalfNormal(1)``.
      * ``"horseshoe"``: regularized horseshoe (Piironen & Vehtari), ``beta_k = z_k * tau * lam~_k`` with
        ``z_k ~ N(0, 1)``, ``tau ~ HalfCauchy(0.1)``, ``lam_k ~ HalfCauchy(1)`` and a slab ``c^2 ~ InvGamma(2, 2)``.
      * ``"gaussian"``: ``beta_k ~ N(0, 0.1^2)``, as in ``SspReg`` (control).
    """
    B, n_states, K = y.shape[0], Z.shape[1], X.shape[1]
    with numpyro.plate("series", B):
        sigma_h = numpyro.sample("sigma_h", sigma_h_prior.distribution)
        sigma_q_level = numpyro.sample("sigma_q_level", sigma_q_prior.distribution)
        sigma_q_dow = numpyro.sample("sigma_q_dow", sigma_q_prior.distribution)
        if prior == "laplace":
            b = numpyro.sample("b", dist.HalfNormal(1.0))
            u = numpyro.sample("u", dist.Laplace(0.0, 1.0).expand([K]).to_event(1))
            beta = numpyro.deterministic("beta", b[:, None] * u)
        elif prior == "horseshoe":
            tau = numpyro.sample("tau", dist.HalfCauchy(0.1))
            lam = numpyro.sample("lam", dist.HalfCauchy(1.0).expand([K]).to_event(1))
            c2 = numpyro.sample("c2", dist.InverseGamma(2.0, 2.0))
            z = numpyro.sample("z", dist.Normal(0.0, 1.0).expand([K]).to_event(1))
            lam_tilde = jnp.sqrt(c2[:, None] * lam**2 / (c2[:, None] + tau[:, None] ** 2 * lam**2))
            beta = numpyro.deterministic("beta", z * tau[:, None] * lam_tilde)
        else:
            beta = numpyro.sample("beta", dist.Normal(0.0, 0.1).expand([K]).to_event(1))
        y = y - beta @ X.T
    sigma_q = jnp.column_stack([sigma_q_level, jnp.broadcast_to(sigma_q_dow[:, None], (B, n_states - 1))])
    log_p, *_ = kalman_filter_1d_batch(
        a0=jnp.zeros(n_states),
        P0=jnp.ones(n_states),
        Z=Z,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        logp=True,
        chunk_size=None,
    )
    numpyro.factor("log_lik", log_p.sum())


def fit_sparse_nuts(prior, chunk=10, num_warmup=200, num_samples=200, num_chains=4, target_accept=0.8, seed=0):
    """NUTS fit of the subset in runs of ``chunk`` series, like ``SspReg``. Returns forecasts and diagnostics."""
    params_t = transform.fit(sales)
    y_all = np.asarray(transform.forward(sales, params_t))
    Zj, Xj = jnp.asarray(Z, dtype=jnp.float32), jnp.asarray(X, dtype=jnp.float32)
    forecasts, betas, n_div = [], [], 0
    for start in range(0, n_series, chunk):
        sl = slice(start, start + chunk)
        mcmc = MCMC(
            NUTS(ssp_sparse_model, target_accept_prob=target_accept),
            num_warmup=num_warmup,
            num_samples=num_samples,
            num_chains=num_chains,
            chain_method="vectorized",
            progress_bar=False,
        )
        mcmc.run(
            jax.random.PRNGKey(seed),
            jnp.asarray(y_all[sl]),
            Zj,
            Xj,
            prior,
            HALF_NORMAL_PRIORS["sigma_h_prior"],
            HALF_NORMAL_PRIORS["sigma_q_prior"],
            extra_fields=("diverging",),
        )
        n_div += int(np.asarray(mcmc.get_extra_fields()["diverging"]).sum())
        samples = {k: np.asarray(v) for k, v in mcmc.get_samples().items()}
        res = {
            "samples": samples,
            "y": y_all[sl],
            "transform_params": {k: v[sl] for k, v in params_t.items()},
            "transform": transform,
            "Z": Zj,
            "X": np.asarray(Xj),
        }
        forecasts.append(predict_batch_series_reg(res, Z_future, X_future, n_draws=100))
        betas.append(samples["beta"])  # (draws, chunk, K)
    return np.concatenate(forecasts), {"beta": np.concatenate(betas, axis=1), "n_divergences": n_div}


def selected_share(beta_draws, level=0.9):
    """Share of coefficients whose central credible interval excludes 0, per series averaged: a sparsity read-out."""
    lo, hi = np.quantile(beta_draws, [(1 - level) / 2, 1 - (1 - level) / 2], axis=0)
    return float(((lo > 0) | (hi < 0)).sum(axis=1).mean())

In [ ]:
nuts_out = {}
for prior in ("laplace", "horseshoe"):
    t0 = time.time()
    f, info = fit_sparse_nuts(prior)
    t = time.time() - t0
    nuts_out[prior] = (f, info)
    print(
        record(f"NUTS, {prior} prior", f, t, nonzero=selected_share(info["beta"]), n_divergences=info["n_divergences"])
    )

In [ ]:
def fit_sparse_svi(prior, n_steps=3000, lr=1e-2, n_draws=200, seed=0):
    """Mean-field SVI (``AutoNormal``) of the whole batch in one run, then forecasts from posterior draws."""
    params_t = transform.fit(sales)
    y = jnp.asarray(transform.forward(sales, params_t))
    Zj, Xj = jnp.asarray(Z, dtype=jnp.float32), jnp.asarray(X, dtype=jnp.float32)
    args = (y, Zj, Xj, prior, HALF_NORMAL_PRIORS["sigma_h_prior"], HALF_NORMAL_PRIORS["sigma_q_prior"])
    guide = AutoNormal(ssp_sparse_model)
    svi = SVI(ssp_sparse_model, guide, numpyro.optim.Adam(lr), Trace_ELBO())
    result = svi.run(jax.random.PRNGKey(seed), n_steps, *args, progress_bar=False)
    sites = ("sigma_h", "sigma_q_level", "sigma_q_dow", "beta")
    draws = Predictive(ssp_sparse_model, guide=guide, params=result.params, num_samples=n_draws, return_sites=sites)(
        jax.random.PRNGKey(seed + 1), *args
    )
    samples = {k: np.asarray(v) for k, v in draws.items()}
    res = {
        "samples": samples,
        "y": np.asarray(y),
        "transform_params": params_t,
        "transform": transform,
        "Z": Zj,
        "X": np.asarray(Xj),
    }
    return predict_batch_series_reg(res, Z_future, X_future, n_draws=100), {
        "beta": samples["beta"],
        "losses": np.asarray(result.losses),
    }


svi_out = {}
for prior in ("laplace", "horseshoe"):
    t0 = time.time()
    f, info = fit_sparse_svi(prior)
    t = time.time() - t0
    svi_out[prior] = (f, info)
    print(
        record(
            f"SVI (AutoNormal), {prior} prior",
            f,
            t,
            nonzero=selected_share(info["beta"]),
            final_elbo_loss=float(info["losses"][-1]),
        )
    )

## All approaches, side by side

In [ ]:
results = pd.DataFrame(RESULTS).set_index("approach")
results["vs_default"] = results["wrmsse"] - results.loc["#118 default (MAP, no covariates)", "wrmsse"]
results = results.rename(columns={"nonzero": "kept"})
show = ["wrmsse", "vs_default", *[c for c in results.columns if c.startswith("wrmsse_")], "seconds", "kept"]
print(f"Pre-validation WRMSSE of the 3-item x 10-store subset ({K} covariates)")
print("kept = coefficients per series: non-zero for the MAP fits, with a 90% interval that excludes 0 for NUTS and SVI")
print(results[show].round(4).to_string())

In [ ]:
# Posterior-mean (or MAP) coefficients of one series per approach, by covariate group: which columns each prior keeps.
which = 0  # the first series of the subset: the high-volume item in the first store
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
coefs = {
    "MAP L1": prox_rows["L1"][1]["samples"]["beta"][0][which],
    "MAP group L1": prox_rows["group L1"][1]["samples"]["beta"][0][which],
    "NUTS Laplace": nuts_out["laplace"][1]["beta"].mean(axis=0)[which],
    "NUTS horseshoe": nuts_out["horseshoe"][1]["beta"].mean(axis=0)[which],
}
for ax, (name, beta) in zip(axes, coefs.items()):
    ax.bar(np.arange(K), beta, color=[f"C{list(GROUPS).index(g)}" for g in group_of])
    ax.set_title(name)
    ax.set_xlabel("covariate column")
axes[0].set_ylabel("coefficient")
fig.suptitle(
    f"Coefficients of {base_ids[which]}  (colours: " + ", ".join(f"C{i} = {g}" for i, g in enumerate(GROUPS)) + ")"
)
plt.tight_layout()
plt.show()

The coefficients of the first series (the high item in the first store) under each sparse approach, coloured by covariate
group: which columns each prior keeps and how hard it shrinks the rest. The MAP fits show exact zeros; the NUTS fits show posterior means.

## Findings

Numbers are from one run of this notebook on the 30-series subset (pre-validation unless said otherwise; WRMSSE, lower is
better; "vs default" is the change from the #118 default, 0.5385).

| approach | WRMSSE | vs default | fit time | coefficients kept per series |
|---|---|---|---|---|
| #118 default (MAP, no covariates) | 0.5385 | | 1.2 s | 0 |
| `SspReg`, Gaussian `tau` = 1 (NUTS) | 0.5355 | -0.0030 | 596 s | 73 |
| `SspReg`, Gaussian `tau` = 0.1 (NUTS) | 0.5341 | -0.0044 | 175 s | 73 |
| MAP, no penalty | 0.5254 | -0.0131 | 1.7 s | 73 |
| MAP, Gaussian `tau` = 1 (ridge) | 0.5227 | -0.0158 | 1.7 s | 73 |
| MAP + L1 prox, `lambda` = 3 (holdout choice) | 0.5288 | -0.0097 | 1.7 s (+36 s to choose `lambda`) | 41 |
| MAP + group L1 prox, `lambda` = 0.3 (holdout choice) | 0.5340 | -0.0045 | 1.6 s (+36 s) | 72 |
| NUTS, Laplace prior | 0.5383 | -0.0002 | 2,267 s | 4.5 (90% interval excludes 0) |
| NUTS, regularized horseshoe prior | 0.5441 | +0.0056 | 1,557 s, 24 divergences | 2.8 (90% interval excludes 0) |
| SVI, Laplace prior | 0.5504 | +0.0119 | 17 s | 6.2 (90% interval excludes 0) |
| SVI, regularized horseshoe prior | 0.5613 | +0.0228 | 17 s | 2.5 (90% interval excludes 0) |

1. **Sparsity does not buy accuracy on this subset; the covariates do, as long as they are not shrunk away.** The MAP fit
   with no penalty, with a Gaussian prior and with a light L1 penalty (`lambda` ≤ 3) all beat the default by 0.010-0.021 at
   pre-validation, and by 0.004-0.040 at the validation and evaluation origins (the origins table above). A middle L1 penalty
   (`lambda` = 10-30, 2-16 coefficients left) is *worse* than keeping all or none at pre-validation (0.556 and 0.559 against
   0.538), and at the validation origin (0.5173 and 0.5234 against 0.5152); at the evaluation origin `lambda` = 10 is better
   than none (0.5748 against 0.5914), so the middle of the range is noisy rather than uniformly bad.
   The gain is in the high item (0.534 → 0.516-0.523); the low item does not move, and the mid item gets worse with the
   covariates in the MAP and Gaussian-NUTS fits (0.599 → 0.605-0.640; only the two SVI fits are below the default).
2. **The sampling-based sparse priors land on the baseline, at a cost no full run can pay.** The Laplace posterior shrinks
   the coefficients until the score is the default's (0.5383 against 0.5385), and the horseshoe is worse (0.5441) with 24
   divergent transitions over the 3 runs. They take 26-38 minutes for 30 series: 9-13 times `SspReg` with `tau` = 0.1 and
   about 1,000 times the MAP fit. At 10 series per run, 30,490 series would be 3,049 runs of 9-13 minutes each, some 440-640
   hours on this CPU. This is the speed and mixing problem the issue asked about: the heavy-tailed scales are what NUTS finds hard, and the product of three scales
   (`tau`, `lam_k`, `c`) has to be explored per series.
3. **SVI is fast (17 s for all 30 series) but not good enough:** the mean-field fit gives the two worst rows (+0.012 and
   +0.023), the high item most of all (0.546 and 0.557 against 0.534). It would need a better guide or more steps to be a
   candidate, and a sparse posterior is what a mean-field normal represents badly.
4. **NUTS is also the slow part of the Gaussian baseline:** `SspReg` takes 175 s at `tau` = 0.1 and 596 s at `tau` = 1
   (smallest effective sample size 21 against 188), and the MAP fit of the same model with the same covariates is 100-350
   times faster and scores better (0.5227-0.5278 against 0.5341-0.5355). The posterior mean of `SspReg` is not what is
   helping; a point estimate of `beta` is.

Limits of this evidence: 30 series from 3 items, one 28-day window per origin, the NUTS and SVI fits at the pre-validation
origin only, and one seed. The differences between the MAP rows that are within 0.005 should not be read as a ranking. The
default itself has 0.0004 of slack between two implementations of the same model (the first MAP row against the
`lambda` = infinity control).

### What each approach uses, and what is missing

| approach | numpyro and optax modules | missing piece |
|---|---|---|
| L1 / group L1 proximal MAP (built here) | `optax.adam`; `numpyro.distributions` for the sigma priors | A MAP fitter with a static `X beta` in `m5_ssp_optim.py` (`fit_batch_series_opt` has covariates only as filter states), a proximal step, and the rolling-fold choice of `lambda` |
| Gaussian MAP, no penalty (built here) | the same | the same fitter with `lambda` = 0, `tau` as a Gaussian prior |
| NUTS with a Laplace prior (built here) | `MCMC`, `NUTS`, `distributions.Laplace`, `HalfNormal`, `plate` | nothing missing; too slow |
| NUTS with a regularized horseshoe (built here) | `MCMC`, `NUTS`, `HalfCauchy`, `InverseGamma`; `reparam.LocScaleReparam` for the non-centering | nothing missing; slow, and divergent |
| SVI (built here) | `SVI`, `Trace_ELBO`, `autoguide.AutoNormal`, `Predictive` | nothing missing; a better guide |
| Scale mixture through the Kalman filter (not built) | `NUTS` over `tau`, the `lam_j` and the sigmas | A filter that keeps the covariance between states: `kalman_filter_1d` is diagonal, which #146 and #148 found makes coefficient readings too narrow |
| Gibbs Bayesian LASSO, Park & Casella (not built) | `Gibbs`, `HMCGibbs`, `CustomGibbs` | An inverse-Gaussian sampler (`dist.InverseGaussian` is not in 0.22) and a simulation smoother to draw the coefficients (`kalman_dk_smoother_1d` and `kalman_rts_smoother_1d` return means) |
| Gibbs horseshoe, Makalic & Schmidt (not built) | `HMCGibbs` or `CustomGibbs`, `InverseGamma` | A simulation smoother (or a full-covariance filter) to draw the coefficients given the scales |
| Spike and slab (not built) | `DiscreteHMCGibbs`, `MixedHMC`, `DiscreteGibbs` | One Kalman pass per indicator flip: 73 passes per sweep per series |
| SVI with `AutoLaplaceApproximation` (not built) | `SVI`, `autoguide.AutoLaplaceApproximation` | Defined for a Laplace prior, not for a horseshoe (unbounded density at 0) |

The Gibbs, scale-mixture and spike-and-slab rows are read from the numpyro 0.22 and `bunobee` API (the inventory above), not
run: after the NUTS results they would have to beat a sampler that already matches the baseline, at a cost per sweep that
the table shows is not small.

### Recommendation

**Build the optimisation-only approach as the next M5 entry:** `SspDow`-style MAP with the covariates as a static regression
`X beta` outside the filter (the `SspReg` model with MAP instead of NUTS), with a proximal L1 step and a Gaussian prior as
two settings of the same fitter.

* It is the only approach that is fast (1.7 s for 30 series; choosing `lambda` on 7 values × 3 rolling folds takes another
  36 s, and can be done once and then fixed), and it scores better than the default at all three origins for `lambda` ≤ 3.
* The entry should default to `lambda` = 3 with the plain L1 step, or `tau` = 1 as a ridge if exact zeros are not needed:
  on this evidence they are equal, and L1 buys interpretability (41 of 73 coefficients kept) and a smaller design, not accuracy.
  Group L1 chose a `lambda` that kept 72 of 73 coefficients, so it adds nothing here.
* What it needs: a MAP fitter for `y - X beta` in `src/bunobee/models/m5/m5_ssp_optim.py` (the loss and the prox step of
  `fit_map_l1`, with `beta` as a parameter next to the sigmas), a `SspDow`-like entry in `m5_models.py` with `lambda` and
  `tau` fields, and `predict_batch_series_reg` for the forecast, which already takes one "draw". No numpyro module is
  missing, and neither a full-covariance filter nor a simulation smoother is needed.
* Before it is registered, check it on the 3,000-series sample (not the full run), since a result on 3 items at 3 origins is
  the strongest claim these data support. The mid item gets worse with the covariates in every fit, which a per-tier `lambda`
  may fix.

**Do not build the sampling-based LASSO or horseshoe for M5.** They match the baseline at best, take 26-38 minutes for 30
series, and the Gibbs routes that could make them faster need an inverse-Gaussian sampler and a simulation smoother that
the project does not have. If a posterior over the coefficients is wanted later, the missing pieces are the full-covariance
filter and the simulation smoother, and that is a different issue.